# 11 — Descriptive trade and supply

Describe long-run trade and refinery-output dynamics before any event modelling. Persist every report candidate figure and headline metric.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)


In [ ]:
panel = pd.read_csv(PATHS.processed / "fuel_annual_analytical_panel.csv")
for product in ["diesel", "gasoline"]:
    sub = panel.loc[panel["product"] == product].sort_values("year")
    fig, ax = plt.subplots(figsize=(10, 5))
    for col, label in [("exports_kt", "Exports"), ("imports_kt", "Imports"), ("demand_kt", "Domestic demand"), ("refinery_output_kt", "Refinery output")]:
        if col in sub and sub[col].notna().any():
            ax.plot(sub["year"], sub[col], marker="o", label=label)
    ax.axvline(2013, linestyle="--", linewidth=1, label="Sines hydrocracker")
    ax.axvline(2021, linestyle=":", linewidth=1, label="Matosinhos transition")
    ax.set(title=f"Portugal {product}: physical balance", xlabel="Year", ylabel="Thousand tonnes")
    ax.legend(ncol=2)
    fig.tight_layout()
    fig.savefig(PATHS.figures / f"{product}_physical_balance.png", dpi=180)
    plt.show()


In [ ]:
metric_rows = []
for product, sub in panel.groupby("product"):
    for column in ["exports_kt", "imports_kt", "demand_kt", "refinery_output_kt"]:
        if column not in sub or sub[column].dropna().empty:
            continue
        peak_idx = sub[column].idxmax()
        metric_rows.extend([
            {"metric": f"{column}_mean_2005_2024", "product": product, "value": sub[column].mean(), "unit": "kt"},
            {"metric": f"{column}_peak", "product": product, "value": sub.loc[peak_idx, column], "unit": "kt"},
            {"metric": f"{column}_peak_year", "product": product, "value": sub.loc[peak_idx, "year"], "unit": "year"},
        ])
metrics = pd.DataFrame(metric_rows)
persist_dataframe(metrics, PATHS.metrics / "descriptive_metrics.csv")
display(metrics)
